# setup

## write dsub file

In [ ]:
%%writefile ~/aou_dsub.bash

#!/bin/bash

# This shell function passes reasonable defaults for several dsub parameters, while
# allowing the caller to override any of them. It creates a nice folder structure within
# the workspace bucket for dsub log files.

# --[ Parameters ]--
# any valid dsub parameter flag

#--[ Returns ]--
# the job id of the job created by dsub

#--[ Details ]--
# The first five parameters below should always be those values when running on AoU RWB.

# Feel free to change the values for --user, --regions, --logging, and --image if you like.

# Note that we insert some job data into the logging path.
# https://github.com/DataBiosphere/dsub/blob/main/docs/logging.md#inserting-job-data

function aou_dsub () {

  # Get a shorter username to leave more characters for the job name.
  local DSUB_USER_NAME="$(echo "${OWNER_EMAIL}" | cut -d@ -f1)"

  # For AoU RWB projects network name is "network".
  #local AOU_NETWORK=network
  #local AOU_SUBNETWORK=subnetwork

  dsub \
      --provider google-batch \
      --user-project "${GOOGLE_PROJECT}"\
      --project "${GOOGLE_PROJECT}"\
      --image 'ubuntu:latest' \
      --network "global/networks/network" \
      --subnetwork "regions/us-central1/subnetworks/subnetwork" \
      --service-account "$(gcloud config get-value account)" \
      --use-private-address \
      --user "${DSUB_USER_NAME}" \
      --regions us-central1 \
      --logging "${WORKSPACE_BUCKET}/dsub/logs/{job-name}/{user-id}/$(date +'%Y%m%d/%H%M%S')/{job-id}-{task-id}-{task-attempt}.log" \
      "$@"
}

# compute PGS- HF

## subset plink files to variants in scores and individuals in sample

### copy plink files to workspace bucket

In [ ]:
!gsutil -u $GOOGLE_PROJECT -m cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/pgen/* ${WORKSPACE_BUCKET}/HF/input/acaf/

In [ ]:
!gsutil -u $GOOGLE_PROJECT ls -lh gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/pgen/*

### run command

In [ ]:
import pandas as pd
import os
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
JOB_NAME=f'plink_subset-{USER_NAME}'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--env CHROM': [x for x in range(1, 23)],
    '--output-recursive OUT_DIR': [f"{bucket}/HF/input/acaf_pgen_subset/" for _ in range(22)]
})

PARAMETER_FILENAME = f'{JOB_NAME}_params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index=False)

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-cls-v2 \
  --image "gcr.io/ritchie-aou-psom-9015/plink2:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/plink_subset" \
  --disk-size 180 \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'plink2 --pfile $BUCKET/HF/input/acaf/acaf_threshold.chr$CHROM \
                      --extract range $BUCKET/HF/input/HF.PGS005097.var_list.txt \
                      --set-all-var-ids @:#:\$r:$a \
                      --new-id-max-allele-len 1000 \
                      --make-pgen \
                      --out $OUT_DIR/acaf_threshold.hf_pgs_weights_vars.chr$CHROM'

print("\n".join(job_output))
plink_job_id = job_output[1].split(" ")[-1]
%env PLINK_JOB_ID={plink_job_id}

### check worker status

In [ ]:
!dstat \
    --provider google-cls-v2 \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

### remove plink files from workspace bucket

In [ ]:
!gsutil rm ${WORKSPACE_BUCKET}/HF/input/acaf/*

## create sample sheet

### create file path list

In [ ]:
filepath_list = []
for chr in list(range(1, 23)):
    filepath = '/mnt/data/mount/gs/fc-secure-xxx/HF/input/acaf_pgen_subset/acaf_threshold.hf_pgs_weights_vars.chr' + str(chr)
    filepath_list.append(filepath)
print(len(filepath_list))
filepath_list

### create dataframe

In [ ]:
import pandas as pd
samplesheet = pd.DataFrame(data = {'sampleset' : 'AOU',
                                   'path_prefix' : filepath_list,
                                   'chrom' : list(range(1, 23)),
                                   'format' : 'pfile'})
samplesheet

### export dataframe

In [ ]:
samplesheet.to_csv('AOU.HF.PGSC_CALC.samplesheet.csv', index = None)

### copy to workspace bucket

In [ ]:
!gsutil cp AOU.HF.PGSC_CALC.samplesheet.csv ${WORKSPACE_BUCKET}/HF/input/

## run PGSC-calc

### run command

In [ ]:
import os
import pandas as pd
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
job_output = []


JOB_NAME=f'pgsc_calc'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--output-recursive OUT_DIR': [f"{bucket}/HF/output/"]
})

PARAMETER_FILENAME = f'params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-cls-v2 \
  --image "gcr.io/ritchie-aou-psom-9015/pgsc_calc_general:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/hf/" \
  --disk-size 512 \
  --boot-disk-size 20 \
  --min-ram 208 \
  --min-cores 32 \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'nextflow run pgscatalog/pgsc_calc -profile conda \
                  --input $BUCKET/CKD/input/AOU.CKD.PGSC_CALC.samplesheet.csv \
                  --scorefile "$BUCKET/HF/input/pgs_weights/*" \
                  --target_build GRCh38 \
                  --outdir $OUT_DIR \
                  --max_cpus 32 \
                  --max_memory 208.GB \
                  --min_overlap 0.0 \
                  --max_time 240.h \
                  --run_ancestry $BUCKET/CKD/input/pgsc_HGDP+1kGP_v1.tar.zst \
                  --keep_multiallelic True \
                  --hwe_ref 0 \
                  --pca_maf_target 0.05'

job_id = job_output[1].split(" ")[-1]
%env JOB_ID={job_id}

### check worker status

In [ ]:
!dstat \
    --provider google-cls-v2 \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

### check logs

In [ ]:
!gsutil cp ${WORKSPACE_BUCKET}/dsub_logs/hf/pgsc-calc--kathleen-cardone--250703-010752-01* .

In [ ]:
!cat pgsc-calc--kathleen-cardone--250703-010752-01.1-stderr.log

In [ ]:
!cat pgsc-calc--kathleen-cardone--250703-010752-01.1-stdout.log

# Compute PGS- CAD and AFIB

## subset plink files

### copy plink files to workspace bucket

In [ ]:
!gsutil -u $GOOGLE_PROJECT -m cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/pgen/* ${WORKSPACE_BUCKET}/CAD_AFIB/input/acaf/

### submit job

In [ ]:
import pandas as pd
import os
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
JOB_NAME=f'plink_subset-{USER_NAME}'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--env CHROM': [x for x in range(1, 23)],
    '--output-recursive OUT_DIR': [f"{bucket}/CAD_AFIB/input/acaf_pgen_subset/" for _ in range(22)]
})

PARAMETER_FILENAME = f'{JOB_NAME}_params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-batch \
  --image "gcr.io/ritchie-aou-psom-9015/plink2:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/cad_afib/plink_subset" \
  --disk-size 180 \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'plink2 --pfile $BUCKET/CAD_AFIB/input/acaf/acaf_threshold.chr$CHROM \
                      --extract range $BUCKET/CAD_AFIB/input/CAD.PGS005112.AFIB.PGS005072.eMERGE_weights.var_list.txt \
                      --set-all-var-ids @:#:\$r:$a \
                      --new-id-max-allele-len 1000 \
                      --make-pgen \
                      --out $OUT_DIR/acaf_threshold.cad_afib_pgs_weights_vars.chr$CHROM'

print("\n".join(job_output))
plink_job_id = job_output[1].split(" ")[-1]
%env PLINK_JOB_ID={plink_job_id}

### check status

In [ ]:
!dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

### check logs

In [ ]:
!gsutil cp ${WORKSPACE_BUCKET}/dsub_logs/cad_afib/plink_subset/plink-subs--kathleen-cardone--251104-173645-45.1-* .

In [ ]:
!cat plink-subs--kathleen-cardone--251104-173645-45.1-stderr.log

### remove plink files from workspace bucket

In [ ]:
!gsutil rm ${WORKSPACE_BUCKET}/CAD_AFIB/input/acaf/*

## create samplesheet

### make filepath list

In [ ]:
filepath_list = []
for chr in list(range(1, 23)):
    filepath = '/mnt/data/mount/gs/fc-secure-xxx/CAD_AFIB/input/acaf_pgen_subset/acaf_threshold.cad_afib_pgs_weights_vars.chr' + str(chr)
    filepath_list.append(filepath)
print(len(filepath_list))

### create df

In [ ]:
import pandas as pd
samplesheet = pd.DataFrame(data = {'sampleset' : 'AOU',
                                   'path_prefix' : filepath_list,
                                   'chrom' : list(range(1, 23)),
                                   'format' : 'pfile'})
samplesheet.head()

### export and copy to bucket

In [ ]:
samplesheet.to_csv('AOU.CAD_AFIB.PGSC_CALC.samplesheet.csv', index = None)

In [ ]:
!gsutil cp AOU.CAD_AFIB.PGSC_CALC.samplesheet.csv ${WORKSPACE_BUCKET}/CAD_AFIB/input/

## run PGSC calc

### submit job

In [ ]:
import os
import pandas as pd
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
job_output = []


JOB_NAME=f'pgsc_calc'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--output-recursive OUT_DIR': [f"{bucket}/CAD_AFIB/output/"]
})

PARAMETER_FILENAME = f'params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-batch \
  --image "gcr.io/ritchie-aou-psom-9015/pgsc_calc_general:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/cad_afib/pgsc_calc/" \
  --disk-size 512 \
  --boot-disk-size 20 \
  --min-ram 208 \
  --min-cores 32 \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'nextflow run pgscatalog/pgsc_calc -profile conda \
                  --input $BUCKET/CAD_AFIB/input/AOU.CAD_AFIB.PGSC_CALC.samplesheet.csv \
                  --scorefile "$BUCKET/CAD_AFIB/input/pgs_weights/*" \
                  --target_build GRCh38 \
                  --outdir $OUT_DIR \
                  --max_cpus 32 \
                  --max_memory 208.GB \
                  --min_overlap 0.0 \
                  --max_time 240.h \
                  --run_ancestry $BUCKET/CKD/input/pgsc_HGDP+1kGP_v1.tar.zst \
                  --keep_multiallelic True \
                  --hwe_ref 0 \
                  --pca_maf_target 0.05'

job_id = job_output[1].split(" ")[-1]
%env JOB_ID={job_id}

### check job status

In [ ]:
!dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

### check logs

In [ ]:
!gsutil cp ${WORKSPACE_BUCKET}/dsub_logs/cad_afib/pgsc_calc/pgsc-calc--kathleen-cardone--260217-170309-80.1-* .

In [ ]:
!cat pgsc-calc--kathleen-cardone--260217-170309-80.1-stdout.log

# run catboost feature selection script

## upload script to bucket

In [ ]:
!gsutil cp catboost_feature_selection.py ${WORKSPACE_BUCKET}/CAD_AFIB/scripts/

## upload inputs to workspace bucket

In [ ]:
!gsutil cp CAD.IRM.PXS_feature_selection_input.csv ${WORKSPACE_BUCKET}/CAD_AFIB/input/

In [ ]:
!gsutil cp AFIB.IRM.PXS_feature_selection_input.csv ${WORKSPACE_BUCKET}/CAD_AFIB/input/

In [ ]:
!gsutil cp HF.IRM.PXS_feature_selection_input.csv ${WORKSPACE_BUCKET}/CAD_AFIB/input/

## CAD

### remove old files if running again

In [ ]:
!gsutil rm -R ${WORKSPACE_BUCKET}/CAD_AFIB/output/CAD/feature_selection/catboost/

### first 500 jobs

#### submit first 500 CAD jobs

In [ ]:
import os
import pandas as pd
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
job_output = []


JOB_NAME=f'catboost'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--env ITER' : list(range(1, 501)),
    '--output-recursive OUT_DIR' : [f"{bucket}/CAD_AFIB/output/CAD/feature_selection/catboost/" for _ in range(500)]
})

PARAMETER_FILENAME = f'params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

print('submitting job')

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-batch \
  --image "katiecardone26/python:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/cad_afib/feature_selection/catboost/" \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'python "${BUCKET}"/CAD_AFIB/scripts/catboost_feature_selection.py \
                --input "${BUCKET}"/CAD_AFIB/input/CAD.IRM.PXS_feature_selection_input.csv \
                --iter "${ITER}" \
                --pheno CAD \
                --output_dir "${OUT_DIR}"/'

job_id = job_output[1].split(" ")[-1]
print(job_id)
%env JOB_ID={job_id}

#### check status

In [ ]:
!dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --users "${USER_NAME}" \
    --jobs "${JOB_ID}" \
    --status '*'

#### check number of output files

In [ ]:
!gsutil ls ${WORKSPACE_BUCKET}/CAD_AFIB/output/CAD/feature_selection/catboost/ | wc -l

### second 500 jobs

#### submit second 500 cad jobs

In [ ]:
import os
import pandas as pd
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
job_output = []


JOB_NAME=f'catboost'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--env ITER' : list(range(501, 1001)),
    '--output-recursive OUT_DIR' : [f"{bucket}/CAD_AFIB/output/CAD/feature_selection/catboost/" for _ in range(500)]
})

PARAMETER_FILENAME = f'params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

print('submitting job')

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-batch \
  --image "katiecardone26/python:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/cad_afib/feature_selection/catboost/" \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'python "${BUCKET}"/CAD_AFIB/scripts/catboost_feature_selection.py \
                --input "${BUCKET}"/CAD_AFIB/input/CAD.IRM.PXS_feature_selection_input.csv \
                --iter "${ITER}" \
                --pheno CAD \
                --output_dir "${OUT_DIR}"/'

job_id = job_output[1].split(" ")[-1]
print(job_id)
%env JOB_ID={job_id}

#### check status

In [ ]:
!dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

#### check number of output files

In [ ]:
!gsutil ls ${WORKSPACE_BUCKET}/CAD_AFIB/output/CAD/feature_selection/catboost/ | wc -l

### resubmit failed jobs

#### identify jobs that failed

In [ ]:
!gsutil ls ${WORKSPACE_BUCKET}/CAD_AFIB/output/CAD/feature_selection/catboost/CAD.catboost.gain* > present.txt
!sort present.txt > present_sorted.txt
!wc -l present_sorted.txt

In [ ]:
%%bash 
for i in {1..1000}; do
  printf "gs://fc-secure-xxx/CAD_AFIB/output/CAD/feature_selection/catboost/CAD.catboost.gain.feature_importance.ITER_%d.txt\n" "$i"
done > expected.txt
sort expected.txt > expected_sorted.txt
wc -l expected_sorted.txt

In [ ]:
!comm -23 expected_sorted.txt present_sorted.txt > missing.txt
!wc -l missing.txt
!cat missing.txt

#### resubmit jobs that failed

In [ ]:
import os
import pandas as pd
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
job_output = []


JOB_NAME=f'catboost'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--env ITER' : [471, 623, 797, 818, 918],
    '--output-recursive OUT_DIR' : [f"{bucket}/CAD_AFIB/output/CAD/feature_selection/catboost/" for _ in range(5)]
})

PARAMETER_FILENAME = f'params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

print('submitting job')

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-batch \
  --image "katiecardone26/python:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/cad_afib/feature_selection/catboost/" \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'python "${BUCKET}"/CAD_AFIB/scripts/catboost_feature_selection.py \
                --input "${BUCKET}"/CAD_AFIB/input/CAD.IRM.PXS_feature_selection_input.csv \
                --iter "${ITER}" \
                --pheno CAD \
                --output_dir "${OUT_DIR}"/'

job_id = job_output[1].split(" ")[-1]
print(job_id)
%env JOB_ID={job_id}

#### check status

In [ ]:
!dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

#### check number of output files

In [ ]:
!gsutil ls ${WORKSPACE_BUCKET}/CAD_AFIB/output/CAD/feature_selection/catboost/ | wc -l

## afib

### first 500 jobs

#### submit first 500 jobs

In [ ]:
!gsutil rm -R ${WORKSPACE_BUCKET}/CAD_AFIB/output/AFIB/feature_selection/catboost/

In [ ]:
import os
import pandas as pd
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
job_output = []


JOB_NAME=f'catboost'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--env ITER' : list(range(1, 501)),
    '--output-recursive OUT_DIR' : [f"{bucket}/CAD_AFIB/output/AFIB/feature_selection/catboost/" for _ in range(500)]
})

PARAMETER_FILENAME = f'params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

print('submitting job')

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-batch \
  --image "katiecardone26/python:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/cad_afib/feature_selection/catboost/" \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'python "${BUCKET}"/CAD_AFIB/scripts/catboost_feature_selection.py \
                --input "${BUCKET}"/CAD_AFIB/input/AFIB.IRM.PXS_feature_selection_input.csv \
                --iter "${ITER}" \
                --pheno AFIB \
                --output_dir "${OUT_DIR}"/'

job_id = job_output[1].split(" ")[-1]
print(job_id)
%env JOB_ID={job_id}

#### check status

In [ ]:
!dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

#### check number of output files

In [ ]:
!gsutil ls ${WORKSPACE_BUCKET}/CAD_AFIB/output/AFIB/feature_selection/catboost/ | wc -l

### second 500 jobs

#### submit second 500 jobs

In [ ]:
import os
import pandas as pd
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
job_output = []


JOB_NAME=f'catboost'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--env ITER' : list(range(501, 1001)),
    '--output-recursive OUT_DIR' : [f"{bucket}/CAD_AFIB/output/AFIB/feature_selection/catboost/" for _ in range(500)]
})

PARAMETER_FILENAME = f'params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

print('submitting job')

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-batch \
  --image "katiecardone26/python:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/cad_afib/feature_selection/catboost/" \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'python "${BUCKET}"/CAD_AFIB/scripts/catboost_feature_selection.py \
                --input "${BUCKET}"/CAD_AFIB/input/AFIB.IRM.PXS_feature_selection_input.csv \
                --iter "${ITER}" \
                --pheno AFIB \
                --output_dir "${OUT_DIR}"/'

job_id = job_output[1].split(" ")[-1]
print(job_id)
%env JOB_ID={job_id}

#### check status

In [ ]:
!dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

#### check number of output files

In [ ]:
!gsutil ls ${WORKSPACE_BUCKET}/CAD_AFIB/output/AFIB/feature_selection/catboost/ | wc -l

## hf

### remove old output files if submitting again

In [ ]:
!gsutil rm -R ${WORKSPACE_BUCKET}/CAD_AFIB/output/HF/feature_selection/catboost/

### first 500 jobs

#### submit first 500 jobs

In [ ]:
import os
import pandas as pd
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
job_output = []


JOB_NAME=f'catboost'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--env ITER' : list(range(1, 501)),
    '--output-recursive OUT_DIR' : [f"{bucket}/CAD_AFIB/output/HF/feature_selection/catboost/" for _ in range(500)]
})

PARAMETER_FILENAME = f'params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

print('submitting job')

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-batch \
  --image "katiecardone26/python:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/cad_afib/feature_selection/catboost/" \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'python "${BUCKET}"/CAD_AFIB/scripts/catboost_feature_selection.py \
                --input "${BUCKET}"/CAD_AFIB/input/HF.IRM.PXS_feature_selection_input.csv \
                --iter "${ITER}" \
                --pheno HF \
                --output_dir "${OUT_DIR}"/'

job_id = job_output[1].split(" ")[-1]
print(job_id)
%env JOB_ID={job_id}

#### check status

In [ ]:
!dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

#### check number of files

In [ ]:
!gsutil ls ${WORKSPACE_BUCKET}/CAD_AFIB/output/HF/feature_selection/catboost/ | wc -l

### second 500 jobs

#### submit second 500 jobs

In [ ]:
import os
import pandas as pd
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
job_output = []


JOB_NAME=f'catboost'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--env ITER' : list(range(501, 1001)),
    '--output-recursive OUT_DIR' : [f"{bucket}/CAD_AFIB/output/HF/feature_selection/catboost/" for _ in range(500)]
})

PARAMETER_FILENAME = f'params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

print('submitting job')

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-batch \
  --image "katiecardone26/python:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/cad_afib/feature_selection/catboost/" \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'python "${BUCKET}"/CAD_AFIB/scripts/catboost_feature_selection.py \
                --input "${BUCKET}"/CAD_AFIB/input/HF.IRM.PXS_feature_selection_input.csv \
                --iter "${ITER}" \
                --pheno HF \
                --output_dir "${OUT_DIR}"/'

job_id = job_output[1].split(" ")[-1]
print(job_id)
%env JOB_ID={job_id}

#### check status

In [ ]:
!dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

#### check number of output files

In [ ]:
!gsutil ls ${WORKSPACE_BUCKET}/CAD_AFIB/output/HF/feature_selection/catboost/ | wc -l

### resubmit failed jobs

#### identify jobs that failed

In [ ]:
!gsutil ls ${WORKSPACE_BUCKET}/CAD_AFIB/output/HF/feature_selection/catboost/HF.catboost.gain* > present.txt
!sort present.txt > present_sorted.txt
!wc -l present_sorted.txt

In [ ]:
%%bash 
for i in {1..1000}; do
  printf "gs://fc-secure-xxx/CAD_AFIB/output/HF/feature_selection/catboost/HF.catboost.gain.feature_importance.ITER_%d.txt\n" "$i"
done > expected.txt
sort expected.txt > expected_sorted.txt
wc -l expected_sorted.txt

In [ ]:
!comm -23 expected_sorted.txt present_sorted.txt > missing.txt
!wc -l missing.txt
!cat missing.txt

#### resubmit jobs that failed

In [ ]:
import os
import pandas as pd
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
job_output = []


JOB_NAME=f'catboost'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--env ITER' : [133, 13, 179, 234, 341, 343, 358, 396, 434, 524, 558, 739, 768, 820],
    '--output-recursive OUT_DIR' : [f"{bucket}/CAD_AFIB/output/HF/feature_selection/catboost/" for _ in range(14)]
})

PARAMETER_FILENAME = f'params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

print('submitting job')

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-batch \
  --image "katiecardone26/python:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/cad_afib/feature_selection/catboost/" \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'python "${BUCKET}"/CAD_AFIB/scripts/catboost_feature_selection.py \
                --input "${BUCKET}"/CAD_AFIB/input/HF.IRM.PXS_feature_selection_input.csv \
                --iter "${ITER}" \
                --pheno HF \
                --output_dir "${OUT_DIR}"/'

job_id = job_output[1].split(" ")[-1]
print(job_id)
%env JOB_ID={job_id}

#### check status

In [ ]:
!dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

#### check number of output files

In [ ]:
!gsutil ls ${WORKSPACE_BUCKET}/CAD_AFIB/output/HF/feature_selection/catboost/ | wc -l

# run evaluation script

## upload script and inputs to bucket

In [ ]:
!gsutil cp IRM_eval_script_missing_data.py ${WORKSPACE_BUCKET}/CAD_AFIB/scripts/

In [ ]:
!gsutil cp CAD.IRM.PREVENT_CRS.no_missing.eval_input.csv ${WORKSPACE_BUCKET}/CAD_AFIB/input/

In [ ]:
!gsutil cp HF.IRM.PREVENT_CRS.no_missing.eval_input.csv ${WORKSPACE_BUCKET}/CAD_AFIB/input/

In [ ]:
!gsutil cp AFIB.IRM.C2HEST_PREVENT_CRS.no_missing.eval_input.csv ${WORKSPACE_BUCKET}/CAD_AFIB/input/

In [ ]:
!gsutil cp AFIB.catboost.shap.selected_features.txt ${WORKSPACE_BUCKET}/CAD_AFIB/input/

In [ ]:
!gsutil cp CAD.catboost.shap.selected_features.txt ${WORKSPACE_BUCKET}/CAD_AFIB/input/

In [ ]:
!gsutil cp HF.catboost.shap.selected_features.txt ${WORKSPACE_BUCKET}/CAD_AFIB/input/

## CAD

### first 500 jobs

#### submit first 500 CAD jobs

In [ ]:
import os
import pandas as pd
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
job_output = []


JOB_NAME=f'eval'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--env ITER': list(range(1, 501)),
    '--output-recursive OUT_DIR': [f"{bucket}/CAD_AFIB/output/CAD/regressions/eval/" for _ in range(500)]
})

PARAMETER_FILENAME = f'params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

print('submitting job')

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-batch \
  --image "katiecardone26/python:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/cad_afib/regressions/" \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'python "${BUCKET}"/CAD_AFIB/scripts/IRM_eval_script_missing_data.py \
                --input "${BUCKET}"/CAD_AFIB/input/CAD.IRM.PREVENT_CRS.no_missing.eval_input.csv \
                --weight "${BUCKET}"/CAD_AFIB/input/CAD.catboost.shap.selected_features.txt \
                --iter "${ITER}" \
                --pheno CAD \
                --output_dir "${OUT_DIR}"/'

job_id = job_output[1].split(" ")[-1]
%env JOB_ID={job_id}
print(job_id)

#### check status

In [ ]:
!dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

#### check number of files

In [ ]:
!gsutil ls ${WORKSPACE_BUCKET}/CAD_AFIB/output/CAD/regressions/eval/ | wc -l

### second 500 jobs

#### submit next 500 CAD jobs

In [ ]:
import os
import pandas as pd
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
job_output = []


JOB_NAME=f'eval'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--env ITER': list(range(501, 1001)),
    '--output-recursive OUT_DIR': [f"{bucket}/CAD_AFIB/output/CAD/regressions/eval/" for _ in range(500)]
})

PARAMETER_FILENAME = f'params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

print('submitting job')

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-batch \
  --image "katiecardone26/python:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/cad_afib/regressions/" \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'python "${BUCKET}"/CAD_AFIB/scripts/IRM_eval_script_missing_data.py \
                --input "${BUCKET}"/CAD_AFIB/input/CAD.IRM.PREVENT_CRS.no_missing.eval_input.csv \
                --weight "${BUCKET}"/CAD_AFIB/input/CAD.catboost.shap.selected_features.txt \
                --iter "${ITER}" \
                --pheno CAD \
                --output_dir "${OUT_DIR}"/'

job_id = job_output[1].split(" ")[-1]
%env JOB_ID={job_id}
print(job_id)

#### check status

In [ ]:
!dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

#### check number of files in output dir

In [ ]:
!gsutil ls ${WORKSPACE_BUCKET}/CAD_AFIB/output/CAD/regressions/eval/ | wc -l

## AFIB

### remove old output files if running again

In [ ]:
!gsutil -m rm -R ${WORKSPACE_BUCKET}/CAD_AFIB/output/AFIB/regressions/eval/

### first 500 jobs

#### submit first 500 AFIB jobs

In [ ]:
import os
import pandas as pd
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
job_output = []


JOB_NAME=f'eval'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--env ITER': list(range(1, 501)),
    '--output-recursive OUT_DIR': [f"{bucket}/CAD_AFIB/output/AFIB/regressions/eval/" for _ in range(500)]
})

PARAMETER_FILENAME = f'params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

print('submitting job')

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-batch \
  --image "katiecardone26/python:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/cad_afib/regressions/" \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'python "${BUCKET}"/CAD_AFIB/scripts/IRM_eval_script_missing_data.py \
                --input "${BUCKET}"/CAD_AFIB/input/AFIB.IRM.C2HEST_PREVENT_CRS.no_missing.eval_input.csv \
                --weight "${BUCKET}"/CAD_AFIB/input/AFIB.catboost.shap.selected_features.txt \
                --iter "${ITER}" \
                --pheno AFIB \
                --output_dir "${OUT_DIR}"/'

job_id = job_output[1].split(" ")[-1]
%env JOB_ID={job_id}
print(job_id)

#### check status

In [ ]:
!dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

#### check number of files

In [ ]:
!gsutil ls ${WORKSPACE_BUCKET}/CAD_AFIB/output/AFIB/regressions/eval/ | wc -l

### second 500 jobs

#### submit next 500 AFIB jobs

In [ ]:
import os
import pandas as pd
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
job_output = []


JOB_NAME=f'eval'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--env ITER': list(range(501, 1001)),
    '--output-recursive OUT_DIR': [f"{bucket}/CAD_AFIB/output/AFIB/regressions/eval/" for _ in range(500)]
})

PARAMETER_FILENAME = f'params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

print('submitting job')

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-batch \
  --image "katiecardone26/python:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/cad_afib/regressions/" \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'python "${BUCKET}"/CAD_AFIB/scripts/IRM_eval_script_missing_data.py \
                --input "${BUCKET}"/CAD_AFIB/input/AFIB.IRM.C2HEST_PREVENT_CRS.no_missing.eval_input.csv \
                --weight "${BUCKET}"/CAD_AFIB/input/AFIB.catboost.shap.selected_features.txt \
                --iter "${ITER}" \
                --pheno AFIB \
                --output_dir "${OUT_DIR}"/'

job_id = job_output[1].split(" ")[-1]
%env JOB_ID={job_id}
print(job_id)

#### check status

In [ ]:
!dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

#### check number of files in output dir

In [ ]:
!gsutil ls ${WORKSPACE_BUCKET}/CAD_AFIB/output/AFIB/regressions/eval/ | wc -l

## HF

### remove old outputs if rerunning

In [ ]:
!gsutil -m rm -R ${WORKSPACE_BUCKET}/CAD_AFIB/output/HF/regressions/eval/

### first 500 jobs

#### submit first 500 jobs

In [ ]:
import os
import pandas as pd
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
job_output = []


JOB_NAME=f'eval'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--env ITER': list(range(1, 501)),
    '--output-recursive OUT_DIR': [f"{bucket}/CAD_AFIB/output/HF/regressions/eval/" for _ in range(500)]
})

PARAMETER_FILENAME = f'params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

print('submitting job')

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-batch \
  --image "katiecardone26/python:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/cad_afib/regressions/" \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'python "${BUCKET}"/CAD_AFIB/scripts/IRM_eval_script_missing_data.py \
                --input "${BUCKET}"/CAD_AFIB/input/HF.IRM.PREVENT_CRS.no_missing.eval_input.csv \
                --weight "${BUCKET}"/CAD_AFIB/input/HF.catboost.shap.selected_features.txt \
                --iter "${ITER}" \
                --pheno HF \
                --output_dir "${OUT_DIR}"/'

job_id = job_output[1].split(" ")[-1]
%env JOB_ID={job_id}
print(job_id)

#### check status

In [ ]:
!dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "eval--kathleen-cardone--260302-155102-96" \
    --users "kathleen-cardone" \
    --status '*'

#### check number of output files

In [ ]:
!gsutil ls ${WORKSPACE_BUCKET}/CAD_AFIB/output/HF/regressions/eval/ | wc -l

### second 500 jobs

#### submit next 500 jobs

In [ ]:
import os
import pandas as pd
bucket = os.getenv('WORKSPACE_BUCKET')
USER_NAME = os.getenv('OWNER_EMAIL').split('@')[0].replace('.', '-')
%env USER_NAME={USER_NAME}
job_output = []


JOB_NAME=f'eval'
%env JOB_NAME={JOB_NAME}

params_df = pd.DataFrame(data = {
    '--env ITER': list(range(501, 1001)),
    '--output-recursive OUT_DIR': [f"{bucket}/CAD_AFIB/output/HF/regressions/eval/" for _ in range(500)]
})

PARAMETER_FILENAME = f'params.tsv'
%env PARAMETER_FILENAME={PARAMETER_FILENAME}

params_df.to_csv(PARAMETER_FILENAME, sep = '\t', index = False)

print('submitting job')

job_output = !source ~/aou_dsub.bash; aou_dsub \
  --name "${JOB_NAME}" \
  --provider google-batch \
  --image "katiecardone26/python:latest" \
  --logging "${WORKSPACE_BUCKET}/dsub_logs/cad_afib/regressions/" \
  --mount BUCKET="${WORKSPACE_BUCKET}" \
  --tasks "${PARAMETER_FILENAME}" \
  --command 'python "${BUCKET}"/CAD_AFIB/scripts/IRM_eval_script_missing_data.py \
                --input "${BUCKET}"/CAD_AFIB/input/HF.IRM.PREVENT_CRS.no_missing.eval_input.csv \
                --weight "${BUCKET}"/CAD_AFIB/input/HF.catboost.shap.selected_features.txt \
                --iter "${ITER}" \
                --pheno HF \
                --output_dir "${OUT_DIR}"/'

job_id = job_output[1].split(" ")[-1]
%env JOB_ID={job_id}
print(job_id)

#### check status

In [ ]:
!dstat \
    --provider google-batch \
    --project "${GOOGLE_PROJECT}" \
    --location us-central1 \
    --jobs "${JOB_ID}" \
    --users "${USER_NAME}" \
    --status '*'

#### check number of output files

In [ ]:
!gsutil ls ${WORKSPACE_BUCKET}/CAD_AFIB/output/HF/regressions/eval/ | wc -l

# make ROC and PRC curve plots

## make directory if needed

In [ ]:
!mkdir plots

## cad

In [ ]:
%%bash

python IRM_plot_script.py --input CAD.IRM.PREVENT_CRS.no_missing.eval_input.csv \
--pheno CAD \
--weight CAD.catboost.shap.selected_features.txt \
--mean_metrics AOU.CAD.IRM.population_performance.reformat.csv \
--output_dir plots/ \
--iter 1

## AFIB

In [ ]:
%%bash

python IRM_plot_script.py --input AFIB.IRM.C2HEST_PREVENT_CRS.no_missing.eval_input.csv \
--pheno AFIB \
--weight AFIB.catboost.shap.selected_features.txt \
--mean_metrics AOU.AFIB.IRM.population_performance.reformat.csv \
--output_dir plots/ \
--iter 1

## HF

In [ ]:
%%bash

python IRM_plot_script.py --input HF.IRM.PREVENT_CRS.no_missing.eval_input.csv \
--pheno HF \
--weight HF.catboost.shap.selected_features.txt \
--mean_metrics AOU.HF.IRM.population_performance.reformat.csv \
--output_dir plots/ \
--iter 1